# Experiment 1: Confirm Emotion Directions Exist

**Goal:** Establish that Qwen2.5-0.5B-Instruct has separable internal representations of positive and negative emotional states.

**Method:**
1. Load 10 positive / 10 negative sentence pairs
2. Run each through the model and capture activations at every layer via `baukit.TraceDict`
3. Compute mean-difference direction vectors: `joy = mean(pos) - mean(neg)`
4. Visualize with PCA — do the two groups form separable clusters?

**Success criteria:** Two distinct blobs in PCA space.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import torch
import numpy as np
from tqdm.auto import tqdm

from activation_utils import (
    load_model_and_tokenizer,
    get_layer_names,
    extract_activations,
    compute_mean_direction,
    save_direction,
)
from visualization import plot_activation_pca

print("Libraries loaded successfully")

## 1. Load Model & Data

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cpu"
DTYPE = torch.float32

model, tokenizer = load_model_and_tokenizer(MODEL_NAME, device=DEVICE, torch_dtype=DTYPE)
print(f"Model: {MODEL_NAME}")
print(f"Layers: {len(model.model.layers)}")
print(f"Hidden size: {model.config.hidden_size}")

In [ ]:
# Load contrast pairs
with open('../data/contrast_pairs.json') as f:
    data = json.load(f)

positive_texts = data["positive"]
negative_texts = data["negative"]

print(f"Positive samples: {len(positive_texts)}")
print(f"Negative samples: {len(negative_texts)}")
print("\nExample positive:", positive_texts[0])
print("Example negative:", negative_texts[0])

## 2. Extract Activations at All Layers

In [ ]:
layer_names = get_layer_names(model)
print(f"Hooking {len(layer_names)} layers")
print("First 3:", layer_names[:3])
print("Last 3:", layer_names[-3:])

In [ ]:
%%time
# Extract positive activations
print("Extracting positive activations...")
pos_acts = extract_activations(model, tokenizer, positive_texts, layer_names, device=DEVICE)

# Extract negative activations
print("Extracting negative activations...")
neg_acts = extract_activations(model, tokenizer, negative_texts, layer_names, device=DEVICE)

print("Done!")
print(f"Shape per layer: {pos_acts[layer_names[0]].shape}")  # (n_texts, seq_len, hidden_dim)

## 3. Compute Mean-Difference Direction Vectors Per Layer

In [ ]:
directions = {}
norms = []

for name in tqdm(layer_names, desc="Computing directions"):
    direction = compute_mean_direction(pos_acts[name], neg_acts[name], use_last_token=True)
    directions[name] = direction
    norms.append(direction.norm().item())

# Find the layer with the strongest direction signal
best_idx = int(np.argmax(norms))
best_layer = layer_names[best_idx]
best_norm = norms[best_idx]

print(f"\nStrongest direction found at layer {best_layer}")
print(f"Direction norm: {best_norm:.4f}")
print(f"\nTop 5 layers by norm:")
for idx in np.argsort(norms)[-5:][::-1]:
    print(f"  {layer_names[idx]}: {norms[idx]:.4f}")

## 4. Visualize with PCA

In [ ]:
# Plot PCA for the best layer
fig, pca = plot_activation_pca(
    pos_acts[best_layer],
    neg_acts[best_layer],
    title=f"PCA of Last-Token Activations — {best_layer}",
    save_path="../outputs/figures/ex01_pca_best_layer.png",
)
print(f"PCA explained variance: {pca.explained_variance_ratio_}")

In [ ]:
# Also visualize a few other layers for comparison
layers_to_viz = ["model.layers.5", "model.layers.10", "model.layers.15", "model.layers.20"]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, layer_name in zip(axes, layers_to_viz):
    pos = pos_acts[layer_name][:, -1, :].numpy()
    neg = neg_acts[layer_name][:, -1, :].numpy()
    combined = np.concatenate([pos, neg], axis=0)
    labels = ["pos"] * len(pos) + ["neg"] * len(neg)
    
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    proj = pca.fit_transform(combined)
    
    for label, color in [("pos", "green"), ("neg", "red")]:
        mask = np.array(labels) == label
        ax.scatter(proj[mask, 0], proj[mask, 1], label=label, alpha=0.7, color=color, s=80)
    
    ax.set_title(f"{layer_name} (EV: {pca.explained_variance_ratio_[0]:.1%})")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("PCA Across Multiple Layers", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../outputs/figures/ex01_pca_multi_layer.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Save Final Direction Tensors

In [ ]:
# Save the direction for the best layer (this is what Experiment 2 will use)
joy_direction = directions[best_layer]
grief_direction = -joy_direction  # opposite direction

save_direction(joy_direction, "../outputs/directions/joy_direction.pt")
save_direction(grief_direction, "../outputs/directions/grief_direction.pt")

print(f"Saved joy_direction.pt   (shape: {joy_direction.shape}, norm: {joy_direction.norm():.4f})")
print(f"Saved grief_direction.pt (shape: {grief_direction.shape}, norm: {grief_direction.norm():.4f})")
print(f"\nTarget layer for steering: {best_layer}")

In [ ]:
# Also save a metadata file with layer info
import json

meta = {
    "model": MODEL_NAME,
    "best_layer": best_layer,
    "hidden_size": model.config.hidden_size,
    "n_layers": len(model.model.layers),
    "n_positive_samples": len(positive_texts),
    "n_negative_samples": len(negative_texts),
    "direction_norm": float(joy_direction.norm().item()),
    "layer_norms": {name: float(n) for name, n in zip(layer_names, norms)},
}

with open("../outputs/directions/ex01_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved metadata to ../outputs/directions/ex01_metadata.json")

## 6. Sanity Check: Cosine Similarity Between Positive/Negative Centroids

In [ ]:
# Compute centroid similarity at the best layer
pos_centroid = pos_acts[best_layer][:, -1, :].mean(dim=0)
neg_centroid = neg_acts[best_layer][:, -1, :].mean(dim=0)

cos_sim = torch.nn.functional.cosine_similarity(pos_centroid, neg_centroid, dim=0).item()
euclidean = (pos_centroid - neg_centroid).norm().item()

print(f"Cosine similarity between pos/neg centroids: {cos_sim:.4f}")
print(f"Euclidean distance between centroids:       {euclidean:.4f}")
print(f"\nInterpretation:")
if cos_sim < 0.8:
    print("  -> Good separation! Directions are meaningfully different.")
else:
    print("  -> Centroids are quite similar. Consider more samples or a different layer.")

---

## Summary

This notebook extracted emotion direction vectors from Qwen2.5-0.5B-Instruct by:
1. Passing 10 positive and 10 negative sentences through the model
2. Capturing residual-stream activations at all 24 layers
3. Computing mean-difference vectors per layer
4. Identifying the layer with the strongest signal
5. Saving `joy_direction.pt` and `grief_direction.pt` for Experiment 2

**Next step:** `experiment_02_static_steering.ipynb` — inject these vectors during generation and measure output differences.